# ERC circularité · 00 · Démarrage

Ce dossier aide à décider quoi faire des FRU scrapées d'un centre de réparation et des pièces qu'elles contiennent. Pour chaque pièce d'un FRU scrapé, trois options sont comparées :

1. **Réemploi** : le FRU est ouvert avec soin, la pièce est extraite, testée, puis réutilisée en réparation.
2. **Recyclage trié** : la pièce est triée dans le bon flux et vendue au kilo à un recycleur.
3. **Laissée dans le FRU** : le FRU part tel quel chez le recycleur actuel. C'est la situation d'aujourd'hui, et la référence de tous les calculs.

Ce premier notebook exécute tout le calcul et vérifie que le dossier est en état de marche : sources lues, paramètres, résultats produits, anomalies. Il se lit sans connaître le code.

## Ce que contient le dossier

| Élément | Contenu |
|---|---|
| `notebooks/` | Sept notebooks, à lire dans l'ordre. Ils ne calculent rien eux-mêmes : ils lancent le calcul et affichent ses résultats. |
| `erc/` | Le code du calcul, en sept étapes : sources, classement, flux, fiabilité, modèle de Rex, ligne, synthèse. |
| `parametres/` | `ERC_Parametres.xlsx`, le classeur des paramètres à saisir (temps, coûts, prix, délais), et `chemins.json`, l'emplacement des données réelles. |
| `artefacts/` | Tout ce que le calcul produit : tables (CSV), figures (PNG), un classeur Excel par étape, `manifeste.json`, et l'outil des décideurs. |
| `demo/` | La démonstration complète : sources fictives, paramètres fictifs et leurs artefacts. |
| `docs/Modele_mathematique.md` | La formulation du modèle de ligne et du modèle de Rex. |
| `outils/construire_notebooks.py` | Régénère les notebooks. |

| Notebook | Question traitée |
|---|---|
| **00 Démarrage** | Le dossier fonctionne-t-il, sur quelles données, que manque-t-il ? |
| **01 Classement et priorisation** | Quelles pièces consomme-t-on, lesquelles étudier en premier ? |
| **02 Flux d'entrée** | Combien de FRU scrapées par semaine, quelle masse par benne, l'estimation est-elle confirmée par les pesées ? |
| **03 Fiabilité** | Une pièce réparée ou récupérée tient-elle aussi bien qu'une pièce neuve ? |
| **04 Modèle de Rex** | Quelle est la probabilité qu'une pièce extraite d'un FRU scrapé soit réutilisable ? |
| **05 Ligne de démontage et de tri** | Une ligne est-elle rentable, avec quelles pièces et combien d'heures opérateur ? |
| **06 Synthèse décideurs** | Que retenir, que reste-t-il à obtenir, où est l'outil de décision ? |

La cellule suivante exécute les sept étapes (une vingtaine de secondes en démonstration). Chaque notebook commence par la même cellule, limitée à l'étape dont il a besoin.

In [1]:
import sys
from pathlib import Path

# Le dossier est celui qui contient le code (erc/) : le parent de notebooks/.
dossier = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "erc" / "__init__.py").is_file()), Path.cwd().parent)
sys.path.insert(0, str(dossier))
import erc
from erc import affichage as aff

ctx = erc.executer(jusqua="synthese")

ERC circularité — DÉMONSTRATION (données fictives)


  étape sources       3 artefacts → artefacts/00_sources/


  étape classement   13 artefacts → artefacts/01_classement/


  étape flux         16 artefacts → artefacts/02_flux/


  étape fiabilite    11 artefacts → artefacts/03_fiabilite/


  étape rex          13 artefacts → artefacts/04_rex/


  étape ligne        17 artefacts → artefacts/05_ligne/


  étape synthese      5 artefacts → artefacts/06_synthese/


In [2]:
aff.etat(ctx)

DÉMONSTRATION : toutes les données affichées sont fictives. Période : du 01/06/2026 au 01/11/2026, 154 jours extraits. Aucune anomalie de calcul.

## Démonstration ou données réelles

Sans données réelles, le dossier fonctionne en **mode démonstration** : il fabrique des sources entièrement fictives (références du type `FRU-A100-R` et `P-1004`), dans les mêmes formats de fichiers que les sources réelles, et écrit tout dans `demo/`. Le mode est rappelé dans le bandeau ci-dessus, sur chaque figure et dans le manifeste. Les chiffres de démonstration ne décrivent aucun atelier : ils servent à vérifier que tout s'enchaîne et à montrer comment une donnée manquante est signalée.

**Pour travailler sur les données réelles**, exécuter une fois, dans une cellule de ce notebook :

```python
erc.memoriser_chemins(projet=r"C:\chemin\du\dossier\des\extractions", racine=r"C:\chemin\du\dossier\des\referentiels")
```

puis relancer le notebook en entier. `projet` est le dossier qui contient les extractions `ERC_R26_extraction_AAAAMMJJ_AAAAMMJJ.xlsx` et les sous-dossiers `Classification_matieres/`, `Extraction_R26/` et `Acquisition/` (pesées hebdomadaires, audit de démontage). `racine` est le dossier des référentiels (valorisation, fiche des FRU, fichier de fiabilité) ; s'il est omis, c'est le dossier parent de `projet`.

- Les chemins sont notés dans `parametres/chemins.json`, que tous les notebooks relisent. Les données réelles ne sont jamais copiées dans le dossier : elles sont lues là où elles se trouvent. Les résultats vont dans `artefacts/`, le classeur de paramètres dans `parametres/`.
- Les options d'analyse se mémorisent par la même commande, par exemple `erc.memoriser_chemins(projet=..., racine=..., debut="2026-01-01", fin="2026-07-01")` pour borner la période (début inclus, fin exclue), ou `ateliers_prioritaires=[...]` pour changer les ateliers étudiés en premier. **Chaque appel remplace le précédent** : redonner à chaque fois `projet`, `racine` s'il a été donné, et toutes les options voulues.
- Pour revoir la démonstration sans rien effacer, dans une cellule : `ctx = erc.executer(demo=True)`. Les autres notebooks restent sur les données réelles tant que `parametres/chemins.json` existe ; le supprimer ramène tout le dossier à la démonstration.

**Le dépôt du code est public.** Les artefacts réels (`artefacts/`) et le classeur de paramètres (`parametres/`) en sont exclus, mais un notebook exécuté sur données réelles contient ses résultats : ne pas l'enregistrer dans le dépôt. La commande `python outils/construire_notebooks.py --executer` remet les sept notebooks dans leur état de démonstration.

## Périmètre de cette exécution

Mode, période couverte par les extractions, volumes lus, règle qui date un scrap, emplacement des fichiers. **À vérifier en premier** : la période et le nombre de jours extraits, parce que tous les volumes annuels du dossier en découlent.

La ligne « Classeur de paramètres » dit s'il existait au début de cette exécution : « absent » à la toute première, où il vient d'être créé (section suivante). Les emplacements sont abrégés : seul le dernier élément d'un chemin extérieur au dossier est affiché.

In [3]:
aff.notes(ctx, "sources", "Périmètre")

Élément,Valeur
Mode,DÉMONSTRATION (données fictives)
Démonstration,oui
Période,"du 01/06/2026 au 01/11/2026, 154 jours extraits"
Jours extraits,154
Premier jour extrait,01/06/2026
Dernier jour extrait,01/11/2026
Lignes de consommation lues,1 767
Pièces consommées,2 276
FRU scrapées,273
Date retenue pour le scrap,"plus tardive des dates de la ligne (Date creation PO, Date reception, Date diagnostic, Date statut) : à confirmer"


## État des sources

Une ligne par fichier attendu : chargé, absent ou en erreur, et ce à quoi il sert. Aucune source n'est obligatoire : un fichier absent laisse vides les tables qui en dépendent, il n'arrête pas le calcul et n'est jamais remplacé par des zéros.

**Décision éclairée** : quelles extractions ou quels relevés demander avant de se fier aux résultats. Sans pesées, le flux n'est pas vérifié (notebook 02) ; sans le nombre de pièces posées, aucun taux de défaillance (notebook 03) ; sans audit de démontage, la probabilité de réemploi reste une hypothèse (notebook 04).

In [4]:
aff.table(ctx, "sources", "État des sources")

Source,Fichier,État,Lignes,Détail,Sert à
Extraction R26,ERC_R26_extraction_20260831_20261101.xlsx,chargé,737,20260831 → 20261101,"consommations, FRU scrapées"
Extraction R26,ERC_R26_extraction_20260601_20260913.xlsx,chargé,1 030,20260601 → 20260913,"consommations, FRU scrapées"
FRU_info,FRU_info.xlsx,chargé,5,,atelier et famille de chaque FRU
Classification matières,ERC_Classification_Cartes_resultats.xlsx,chargé,18,,"type de déchet, benne, poids unitaire"
Référentiel des grades,ERC_Referentiel_Grades.xlsx,chargé,8,,prix recycleur par catégorie
QueryComponents,ERC_Valorisation_MICV_v1.xlsx,chargé,23,,"nomenclature de consommation, MMICV de repli"
QueryERC_Flow_In_Out,ERC_Valorisation_MICV_v1.xlsx,chargé,6,,poids et description des FRU
Fiche article (vwAllItems),poids_items_vwAllItems.csv,chargé,19,,MMICV unitaire (coût standard)
Fiabilité (cache),fiabilite_ELF_references_ERC.csv,chargé,399,,défaillances précoces par référence
Pièces posées 2022-2023,poses_2022_2023_par_reference.csv,chargé,8,,dénominateur du taux de défaillance


## Le classeur de paramètres

Les extractions ne contiennent ni les temps de démontage, ni les coûts, ni les prix des recycleurs, ni les délais d'approvisionnement. Ces paramètres se saisissent dans **`parametres/ERC_Parametres.xlsx`** (en démonstration : `demo/parametres/ERC_Parametres.xlsx`).

- **Création** : le classeur est créé à la première exécution, pré-rempli avec les FRU scrapées des ateliers prioritaires et, pour chacune, ses pièces de plus forte valeur et une ligne `CARCASSE` (le châssis et le reste du FRU). **Il n'est jamais réécrit ensuite** : ce qui y est saisi reste. Pour repartir d'un gabarit neuf, renommer l'ancien classeur et relancer.
- **Ce qu'on écrit dans une cellule** :
    - `45` : une valeur ;
    - `30|70` : une plage, du plus bas au plus haut (toutes les valeurs également plausibles) ;
    - `30|45|70` : une plage avec sa valeur la plus plausible, dans l'ordre bas, mode, haut ;
    - cellule vide : inconnu. Le calcul cherche la valeur dans les données (poids, prix du référentiel, MMICV), sinon la laisse inconnue. Jamais zéro.
- **Couleurs** : les colonnes à en-tête **gris** (cellules grises) sont des informations tirées des données, pour repère : les chiffres qu'elles contiennent ne sont pas relus par le calcul. Seule exception : les colonnes qui désignent la ligne (`FRU`, `Pièce`, `Paramètre`, `Référence`) sont relues, ne pas les modifier. Les colonnes à en-tête **bleu** (cellules jaune pâle) sont à saisir.
- **Colonne `Source`** : noter d'où vient chaque valeur (devis, audit du jour J, service qui l'a fournie). Dans les onglets `Ligne`, `FRU` et `Pieces`, la source est reprise telle quelle dans la provenance des paramètres (notebook 05). Une ligne n'a qu'une cellule `Source` : elle vaut pour toutes les valeurs saisies sur cette ligne.
- **Onglets** : `Lisez_moi` (mode d'emploi), `Ligne` (paramètres de la ligne et valeurs par défaut), `FRU` (un type de FRU par ligne ; retirer une ligne l'exclut du modèle), `Pieces` (les pièces de chaque FRU), `Supply` (délais d'approvisionnement), `Ateliers` (rattachement manuel d'un FRU à un atelier), `Chiffres_a_verifier` (chiffres annoncés à comparer aux extractions).
- **Après une saisie** : enregistrer le classeur, puis relancer le notebook. Le notebook 05 montre, paramètre par paramètre, ce qui est saisi, tiré des données ou encore manquant.

Les deux cellules suivantes affichent le mode d'emploi contenu dans le classeur et l'onglet `Ligne` tel qu'il est sur le disque. En démonstration, les valeurs sont fictives et quelques cellules sont laissées vides exprès.

In [5]:
aff.classeur(ctx, "Lisez_moi")

Élément,Mode d'emploi
Rôle,"Paramètres que les données ne contiennent pas : temps, coûts, prix, délais. Le calcul lit ce classeur à chaque exécution et ne le réécrit jamais."
Cellule,"Un nombre (45), une plage bas|haut (30|70) ou un triangle bas|mode|haut (30|45|70). Vide = inconnu."
Colonnes grises,"Informations tirées des données, pour repère. Ne pas modifier FRU, Pièce, Paramètre ni Référence : ce sont les clés qui relient chaque ligne aux données."
Colonnes jaunes,"À saisir. Une cellule vide est cherchée dans les données (poids, prix du référentiel, MMICV) puis laissée inconnue."
Source,"Notez d'où vient chaque valeur (devis, audit du JJ/MM, personne). Elle suit la valeur jusqu'à l'outil des décideurs."
Ligne,"Paramètres de la ligne de démontage et de tri, et valeurs par défaut."
FRU,Un FRU scrapé par ligne. Retirer une ligne l'exclut du modèle ; en ajouter une l'y fait entrer.
Pieces,"Les pièces de chaque FRU. La ligne CARCASSE représente le châssis : poids, prix au kilo et temps de tri seulement."
Supply,Délai d'approvisionnement par référence (achats).
Ateliers,Rattachement manuel d'un FRU à un atelier quand FRU_info ne le donne pas.


In [6]:
aff.classeur(ctx, "Ligne")

Paramètre,Libellé,Unité,Valeur,Source,À demander à
cout_installation_eur,"Investissement d'installation de la ligne (postes, outillage antistatique, banc de test)",€,20000|30000|45000,DÉMO : valeur fictive,"Méthodes, devis fournisseurs"
amortissement_ans,Durée d'amortissement de l'installation,ans,5,DÉMO : valeur fictive,Finance
couts_fixes_annuels_eur,"Coûts fixes annuels de la ligne (place, encadrement, étalonnage)",€/an,6 000,DÉMO : valeur fictive,"Responsables d'atelier, finance"
heures_operateur_an,"Heures opérateur disponibles par an, hors plan de charge",h/an,900,DÉMO : valeur fictive,Responsables d'atelier
taux_horaire_eur,Coût horaire d'un opérateur hors plan de charge (défaut de tous les FRU),€/h,38|42|48,DÉMO : valeur fictive,Responsables d'atelier
cout_traitement_eur_kg,"Coût de traitement du recycleur actuel (0 si gratuit, négatif s'il paie)",€/kg,"0,15|0,35",DÉMO : valeur fictive,"Responsable déchets, contrat actuel"
valeur_reemploi_frac,"Valeur d'une pièce réemployée, en fraction du MMICV (défaut des pièces sans valeur saisie)",0 à 1,"0,3|0,5|0,7",DÉMO : valeur fictive,Finance supply chain
valeur_semaine_eur,Valeur d'une semaine de délai d'approvisionnement évitée (défaut de toutes les pièces),€/semaine,"0,5",DÉMO : valeur fictive,"Achats, service"
taux_change_mmicv,Conversion du MMICV vers l'euro (1 si le MMICV est déjà en euros),€ par unité de MMICV,"0,92",DÉMO : valeur fictive,Finance


## Les artefacts produits

Tout ce que le calcul produit est écrit dans le dossier des artefacts, une étape par sous-dossier :

- une **table** = un fichier CSV (séparateur point-virgule, virgule décimale : il s'ouvre directement dans Excel) ;
- un **classeur Excel par étape**, qui réunit ses tables ;
- les **figures** en PNG, les **notes** de calcul en JSON ;
- `manifeste.json`, la liste ci-dessous.

Les notebooks n'affichent que ce qui est dans ce dépôt : **ce qu'on voit à l'écran est ce qui est sur le disque**. Sous chaque table figure le fichier d'où elle vient. Une case vide est une valeur inconnue. Les tables longues sont coupées aux 30 premières lignes ; le fichier, lui, est complet.

In [7]:
aff.manifeste(ctx)

Étape,Type,Nom,Fichier,Lignes,Description
sources,classeur,classeur sources,00_sources/00_sources.xlsx,,Toutes les tables de l'étape « sources »
sources,json,Périmètre,00_sources/Perimetre.json,,"Mode (réel ou démonstration), période couverte, volumes lus, chemins"
sources,table,Contrôles des sources,00_sources/Controles_des_sources.csv,6,"Ce que la lecture a toléré, écarté ou trouvé incohérent, avec le nombre de lignes concernées"
sources,table,État des sources,00_sources/Etat_des_sources.csv,12,"Chaque fichier attendu : chargé, absent ou en erreur, et ce à quoi il sert"
classement,classeur,classeur classement,01_classement/01_classement.xlsx,,Toutes les tables de l'étape « classement »
classement,figure,Pré-sélection,01_classement/figures/Pre-selection.png,,"Chaque point est une pièce : en abscisse le nombre de FRU scrapées qui la contiennent (gisement), en ordonnée son MMICV total con…"
classement,figure,Top MMICV,01_classement/figures/Top_MMICV.png,,Les pièces consommées classées par MMICV total (quantité × coût standard).
classement,json,Notes,01_classement/Notes.json,,
classement,table,Chiffres à vérifier,01_classement/Chiffres_a_verifier.csv,5,
classement,table,Classement MR,01_classement/Classement_MR.csv,9,


## Anomalies

Un calcul qui échoue n'arrête pas le pipeline : il est inscrit ici, et le résultat correspondant reste vide au lieu d'être inventé. **Table vide = tout a été calculé.** Une anomalie se traite avant de lire les résultats de l'étape concernée.

In [8]:
aff.anomalies(ctx)

Étape,Calcul,Erreur


## Ce que les données ne disent pas

À garder en tête dans tous les notebooks ; la table vient de l'étape de synthèse.

- Le **MMICV** est un coût standard : ni un prix de revente, ni la valeur d'une pièce réemployée.
- Le **nombre de pièces par FRU** est inconnu avant un audit de démontage : les extractions ne donnent qu'un minimum.
- Le **fichier de fiabilité** ne contient que des défaillances : il ne donne pas de durée de vie.
- Les **valeurs** annoncées sont des estimations, distinctes de recettes constatées.
- Le **banc d'essai** du modèle de Rex est une simulation.

In [9]:
aff.table(ctx, "synthese", "Limites et lecture")

Élément,À savoir
Mode,DÉMONSTRATION (données fictives)
Période,"du 01/06/2026 au 01/11/2026, 154 jours extraits"
MMICV,"Coût standard : référence provisoire, ni prix de revente ni valeur de réemploi. Devise à confirmer."
Nombre de pièces par FRU,Inconnu sans audit de démontage : le maximum observé sur un PO est un minimum.
Fiabilité,"Le fichier de fiabilité ne contient que des défaillances : il donne la forme et un taux précoce, pas une durée de vie."
Valeurs potentielles,"Estimations, distinctes des recettes réelles. Toute entrée inconnue laisse le résultat vide."
Banc d'essai du modèle de Rex,"Simulation synthétique : démontre la méthode, ne décrit pas l'atelier."


---
Notebook suivant : **01_Classement_et_priorisation**.